### Required columns :
- customer_id
- period
- monthly_budget
- current_spending
- days_remaining
- avg_daily_spending
- income
- previous_month_expense
- budget_usage_percentage
- category_spending
- projected_month_end_spending
- budget_exceeded

In [120]:
import numpy as np 
import pandas as pd 

In [121]:
raw_df = pd.read_csv(r"C:\Users\shali\OneDrive\Desktop\FinanceX\data\FinanceX_transaction.csv")
raw_df.head()

,customer_id,transaction_date,account_type,transaction_amount,transaction_direction,account_balance,merchant_category,emi_amount,transaction_status,channel,transaction_hour
0,CUST015796,2019-01-01,Current,1820.17,Debit,609365.31,Food & Dining,0.00,Success,Branch,15
1,CUST000861,2019-01-01,Current,392.67,Credit,14451.14,Education,0.00,Success,Mobile_App,3
2,CUST076821,2019-01-01,Fixed Deposit,1255.97,Debit,47621.87,Utilities,1433.91,Success,Mobile_App,18
3,CUST054887,2019-01-01,Savings,2580.68,Debit,34467.85,Travel,6280.35,Success,API,8
4,CUST006266,2019-01-01,Fixed Deposit,2573.80,Debit,26617.40,Utilities,0.00,Success,API,14


#### Period 

In [122]:
raw_df['transaction_date'] = pd.to_datetime(raw_df['transaction_date'])
raw_df['period'] = raw_df['transaction_date'].dt.to_period('M')

In [123]:
raw_df[['transaction_date','period']].head()

,transaction_date,period
0,2019-01-01,2019-01
1,2019-01-01,2019-01
2,2019-01-01,2019-01
3,2019-01-01,2019-01
4,2019-01-01,2019-01


#### Current spending

In [124]:
df = raw_df[
    (raw_df["transaction_status"] == "Success") &
    (raw_df["transaction_direction"] == "Debit")
]

df = (df.groupby(['customer_id','period'],as_index=False)['transaction_amount'].sum())
df = df.rename(columns = {"transaction_amount": "current_spending"})

In [125]:
df.head()

,customer_id,period,current_spending
0,CUST000001,2020-03,892.86
1,CUST000001,2021-08,1000.00
2,CUST000001,2023-02,584.18
3,CUST000001,2023-12,470.63
4,CUST000002,2019-11,186.60


#### days_remaning

In [126]:
raw_df['month_end'] = raw_df["transaction_date"] + pd.offsets.MonthEnd(0)

In [128]:
raw_df['days_remaning'] = (raw_df['month_end'] - raw_df['transaction_date']).dt.days
raw_df[["transaction_date", "month_end", "days_remaning"]].head()

,transaction_date,month_end,days_remaning
0,2019-01-01,2019-01-31,30
1,2019-01-01,2019-01-31,30
2,2019-01-01,2019-01-31,30
3,2019-01-01,2019-01-31,30
4,2019-01-01,2019-01-31,30


#### avg_daily_spending

In [129]:
raw_df['days_elapsed'] = (raw_df['transaction_date'].dt.day)
latest_date = (raw_df.groupby(["customer_id", "period"], as_index=False)["transaction_date"].max())
latest_date["days_elapsed"] = latest_date["transaction_date"].dt.day


In [130]:
df = df.merge(
    latest_date[["customer_id", "period", "days_elapsed"]],
    on=["customer_id", "period"],
    how="left"
)

In [131]:
df["avg_daily_spending"] = (
    df["current_spending"] / df["days_elapsed"]
)

#### Income

In [ ]:
income_df = raw_df[
    (raw_df["transaction_status"] == "Success") &
    (raw_df["transaction_direction"] == "Credit")
]

income_df = (
    income_df
    .groupby(["customer_id", "period"], as_index=False)["transaction_amount"].sum()
)
income_df = income_df.rename(
    columns={"transaction_amount": "income"}
)

In [133]:
df = df.merge(
    income_df,
    on=["customer_id", "period"],
    how="left"
)

In [134]:
df["income"] = df["income"].fillna(0)

#### previous_month_expense

In [135]:
monthly_expense = (
    raw_df[
        (raw_df["transaction_status"] == "Success") &
        (raw_df["transaction_direction"] == "Debit")
    ]
    .groupby(["customer_id", "period"], as_index=False)["transaction_amount"]
    .sum()
)

monthly_expense["period"] = monthly_expense["period"] + 1

monthly_expense = monthly_expense.rename(
    columns={"transaction_amount": "previous_month_expense"}
)


In [136]:
df = df.merge(
    monthly_expense,
    on=["customer_id", "period"],
    how="left"
)

In [137]:
df["previous_month_expense"] = (
    df["previous_month_expense"].fillna(0)
)

#### projected_month_end_spending

In [138]:
latest_date = (
    raw_df
    .groupby(["customer_id", "period"], as_index=False)["transaction_date"]
    .max()
)

# Month ke total days
latest_date["days_in_month"] = (
    latest_date["transaction_date"] + pd.offsets.MonthEnd(0)
).dt.day

# Latest transaction date tak elapsed days
latest_date["days_elapsed"] = latest_date["transaction_date"].dt.day

# Remaining days
latest_date["days_remaining"] = (
    latest_date["days_in_month"]
    - latest_date["days_elapsed"]
)

In [139]:
df = df.merge(
    latest_date[
        ["customer_id", "period", "days_remaining"]
    ],
    on=["customer_id", "period"],
    how="left"
)

In [140]:
df["projected_month_end_spending"] = (
    df["current_spending"]
    + (
        df["avg_daily_spending"]
        * df["days_remaining"]
    )
)

#### monthly_budget 

In [141]:
# Unique customer-month combinations
budget_df = (
    raw_df[["customer_id", "period"]]
    .drop_duplicates()
    .sort_values(["customer_id", "period"])
    .reset_index(drop=True)
)

# Monthly budget
budget_df["monthly_budget"] = 20000


#### budget user percentage 

In [143]:
df = df.merge(
    budget_df,
    on=["customer_id", "period"],
    how="left"
)

df["budget_usage_percentage"] = (
    df["current_spending"] / df["monthly_budget"]
) * 100

#### budget remaning

In [144]:
df["budget_remaining"] = (
    df["monthly_budget"] - df["current_spending"]
)

#### budget exceeded

In [145]:
df["budget_exceeded"] = (
    df["current_spending"] > df["monthly_budget"]
).astype(int)

#### Category spending

In [146]:
category_spending = (
    raw_df[
        (raw_df["transaction_status"] == "Success") &
        (raw_df["transaction_direction"] == "Debit")
    ]
    .groupby(
        ["customer_id", "period", "merchant_category"],
        as_index=False
    )["transaction_amount"]
    .sum()
    .rename(columns={"transaction_amount": "category_spending"})
)

In [147]:
category_spending_final = (
    category_spending
    .groupby(
        ["customer_id", "period"],
        as_index=False
    )["category_spending"]
    .sum()
)

In [148]:
df = df.merge(
    category_spending_final,
    on=["customer_id", "period"],
    how="left"
)

df["category_spending"] = df["category_spending"].fillna(0)

#### Final Dataset columns 

In [149]:
classification_df = df[
    [
        "customer_id",
        "period",
        "monthly_budget",
        "current_spending",
        "days_remaining",
        "avg_daily_spending",
        "income",
        "previous_month_expense",
        "budget_usage_percentage",
        "category_spending",
        "projected_month_end_spending",
        "budget_exceeded"
    ]
].copy()

In [151]:
classification_df.head()

,customer_id,period,monthly_budget,current_spending,days_remaining,avg_daily_spending,income,previous_month_expense,budget_usage_percentage,category_spending,projected_month_end_spending,budget_exceeded
0,CUST000001,2020-03,20000,892.86,1,29.762000,0.0,0.0,4.46430,892.86,922.622000,0
1,CUST000001,2021-08,20000,1000.00,29,500.000000,0.0,0.0,5.00000,1000.00,15500.000000,0
2,CUST000001,2023-02,20000,584.18,15,44.936923,0.0,0.0,2.92090,584.18,1258.233846,0
3,CUST000001,2023-12,20000,470.63,15,29.414375,0.0,0.0,2.35315,470.63,911.845625,0
4,CUST000002,2019-11,20000,186.60,0,6.220000,0.0,0.0,0.93300,186.60,186.600000,0


___